In [ ]:
%matplotlib inline

import jax
from jax import numpy as jnp
from matplotlib import pyplot as plt

from xxm import DatasetCollection, plot
from xxm.core.align import align_affine as align_latent
from xxm.lds import GaussianLDS, LDSParams

plot.set_style()

In [ ]:
def make_rotating_lds(
    angle: float = 0.15,
    process_noise: float = 0.01,
    observation_noise: float = 0.1,
) -> GaussianLDS:
    rotation = jnp.array(
        [
            [jnp.cos(angle), -jnp.sin(angle)],
            [jnp.sin(angle), jnp.cos(angle)],
        ]
    )

    return GaussianLDS.from_params(
        LDSParams(
            initial_mean=jnp.array([1.0, 0.0]),
            initial_covariance=0.01 * jnp.eye(2),
            dynamics_coefficients=rotation,
            dynamics_bias=jnp.zeros(2),
            dynamics_covariance=process_noise * jnp.eye(2),
            emissions=(
                {
                    'coefficients': jnp.array(
                        [
                            [1.0, 0.5],
                            [-0.3, 0.8],
                        ]
                    ),
                    'bias': jnp.zeros(2),
                    'covariance': observation_noise * jnp.eye(2),
                },
            ),
        )
    )


true_model = make_rotating_lds()

truth = true_model.sample(key=jax.random.key(0), steps=300)

data = DatasetCollection.from_array(truth.observations.get())


def plot_latent_data(latents, observations):

    _f, axs = plt.subplots(ncols=2, sharex=True, sharey=True)

    ax = axs[0]
    plot.plot_traces_2d(ax, latents)
    ax.set(title='latent states', aspect='equal')
    plot.hide_ticks(ax)

    ax = axs[1]
    ax.plot(observations[:, 0], observations[:, 1], color='xkcd:magenta')
    ax.set(title='observations', aspect='equal')
    plot.hide_ticks(ax)


plot_latent_data(
    truth.latents.get(),
    truth.observations.get(),
)

In [ ]:
inferred = true_model.infer(data)

In [ ]:
latent_posterior = inferred.latent_posterior().get()

plot.plot_posterior_over_time(latent_posterior, truth.latents.get())
plot.plot_occupancy(latent_posterior, truth.latents.get())

In [ ]:
initial_model = GaussianLDS.via_pca(
    jax.random.key(1),
    data,
    latent_dim=2,
).get()


fitted, objective_trace = initial_model.fit(
    data,
    num_iters=100,
)


fitted_model = fitted.model

plot.plot_fit_progress(objective_trace)

In [ ]:
def align_inferred(inferred, true_latents):

    inferred_latents = inferred.latent_posterior().mean

    alignment = align_latent(inferred_latents.get(), true_latents.get())

    return inferred.align(alignment)


initial = initial_model.infer(data)
initial = align_inferred(initial, truth.latents)
fitted = align_inferred(fitted, truth.latents)

initial_model = initial.model
fitted_model = fitted.model

initial_latent = initial.latent_posterior().mean
fitted_latent = fitted.latent_posterior().mean

In [ ]:
def plot_latent_comparison(true_latents, initial, fitted):
    _, axs = plt.subplots(ncols=2)

    for ax, inferred, title in zip(
        axs,
        (initial, fitted),
        ('Initial posterior', 'Fitted posterior'),
    ):
        plot.plot_occupancy(
            inferred.latent_posterior().get(),
            true_latents,
            ax=ax,
        )
        ax.set(title=title)


plot_latent_comparison(truth.latents.get(), initial, fitted)

In [ ]:
def plot_dynamics(true_model, initial_model, fitted_model):
    _, axs = plt.subplots(
        ncols=2,
        sharex='all',
        sharey='all',
    )

    plot.plot_dyn_linear_gaussian_comparison(
        axs[0], true_model.dynamics(), initial_model.dynamics(), color1='xkcd:grey'
    )
    plot.hide_ticks(axs[0])

    plot.plot_dyn_linear_gaussian_comparison(
        axs[1],
        true_model.dynamics(),
        fitted_model.dynamics(),
        color1='xkcd:apple green',
    )
    plot.hide_ticks(axs[1])


plot_dynamics(true_model, initial_model, fitted_model)

In [ ]:
def plot_observations(observations, initial_prediction, fitted_prediction):
    _f, ax = plt.subplots(figsize=(plot.DOUBLE_COLUMN, plot.SINGLE_COLUMN * 0.5))

    ax.plot(observations[:, 0], alpha=0.4, label='observed', color='k')
    ax.plot(
        initial_prediction[:, 0],
        label='initial',
        color='xkcd:apple green',
        linestyle=':',
    )
    ax.plot(fitted_prediction[:, 0], label='fitted', color='xkcd:coral')

    ax.set(
        xlabel='time',
        ylabel='observations',
    )
    ax.legend()


plot_observations(
    truth.observations.get(),
    initial.observation_mean().get(),
    fitted.observation_mean().get(),
)